# **Tiny Shakespeare**




In [ ]:
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")

# Check CUDA (GPU) availability
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    # Set PyTorch device to GPU
    device = torch.device("cuda")

    # Get current GPU info
    gpu_name = torch.cuda.get_device_name(0)

    # Get VRAM capacity in Gigabytes
    free_mem, total_mem = torch.cuda.mem_get_info(0)
    total_vram_gb = total_mem / (1024 ** 3)
    free_vram_gb = free_mem / (1024 ** 3)

    print(f"GPU Device Name: {gpu_name}")
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
    print(f"Free VRAM: {free_vram_gb:.2f} GB")
else:
    device = torch.device("cpu")
    print("WARNING: GPU not available. Running on CPU. (Go to Runtime -> Change runtime type -> T4 GPU)")

print(f"Selected Device: {device}")

Python Version: 3.12.13
PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Device Name: Tesla T4
Total VRAM: 14.56 GB
Free VRAM: 14.46 GB
Selected Device: cuda


In [ ]:
import time

# Create two large 5000x5000 matrices
size = (5000, 5000)

# CPU Test
x_cpu = torch.randn(size)
y_cpu = torch.randn(size)

start_cpu = time.time()
z_cpu = torch.matmul(x_cpu, y_cpu)
cpu_time = time.time() - start_cpu
print(f"CPU Matrix Multiplication Time: {cpu_time:.4f} seconds")

# GPU Test (if CUDA is available)
if torch.cuda.is_available():
    x_gpu = x_cpu.to("cuda")
    y_gpu = y_cpu.to("cuda")

    # Warmup pass
    _ = torch.matmul(x_gpu, y_gpu)
    torch.cuda.synchronize()

    start_gpu = time.time()
    z_gpu = torch.matmul(x_gpu, y_gpu)
    torch.cuda.synchronize()  # Ensure GPU operation finishes before measuring time
    gpu_time = time.time() - start_gpu

    print(f"GPU Matrix Multiplication Time: {gpu_time:.4f} seconds")
    print(f"Speedup: {cpu_time / gpu_time:.2f}x faster on GPU")

CPU Matrix Multiplication Time: 2.0895 seconds
GPU Matrix Multiplication Time: 0.0881 seconds
Speedup: 23.71x faster on GPU


# Data Set Shakespeare

In [2]:
import os
import urllib.request
from collections import Counter

#File URL and Local name

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
FILE_NAME = "input.txt"

# Download the dataset if it doesn't already exist locally
if not os.path.exists(FILE_NAME):
    print("Downloading Tiny Shakespeare dataset...")
    urllib.request.urlretrieve(DATA_URL, FILE_NAME)
    print("Download complete.")
else:
    print("Dataset already present locally.")

# Read the full text into memory
with open(FILE_NAME, 'r', encoding='utf-8') as f:
    text = f.read()



# Dataset statistics
total_chars = len(text)
unique_chars = sorted(list(set(text)))
vocab_size = len(unique_chars)

print("--- DATASET SUMMARY ---")
print(f"Total length (characters): {total_chars:,}")
print(f"Vocabulary size (unique characters): {vocab_size}")
print(f"Unique characters:\n{''.join(unique_chars)!r}\n")

# Display first 300 characters
print("--- FIRST 300 CHARACTERS ---")
print(text[:1000])






Dataset already present locally.
--- DATASET SUMMARY ---
Total length (characters): 1,115,394
Vocabulary size (unique characters): 65
Unique characters:
"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

--- FIRST 300 CHARACTERS ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts u

In [3]:
# Count occurrence frequency for every character
char_counts = Counter(text)

# Print top 5 most common and top 5 least common characters
print("--- TOP 5 MOST COMMON CHARACTERS ---")
for char, count in char_counts.most_common(5):
    print(f"Character: {char!r:<5} | Count: {count:>7,} | Percentage: {count/total_chars * 100:.2f}%")

print("\n--- TOP 5 LEAST COMMON CHARACTERS ---")
for char, count in char_counts.most_common()[:-6:-1]:
    print(f"Character: {char!r:<5} | Count: {count:>7,} | Percentage: {count/total_chars * 100:.4f}%")

--- TOP 5 MOST COMMON CHARACTERS ---
Character: ' '   | Count: 169,892 | Percentage: 15.23%
Character: 'e'   | Count:  94,611 | Percentage: 8.48%
Character: 't'   | Count:  67,009 | Percentage: 6.01%
Character: 'o'   | Count:  65,798 | Percentage: 5.90%
Character: 'a'   | Count:  55,507 | Percentage: 4.98%

--- TOP 5 LEAST COMMON CHARACTERS ---
Character: '$'   | Count:       1 | Percentage: 0.0001%
Character: '&'   | Count:       3 | Percentage: 0.0003%
Character: '3'   | Count:      27 | Percentage: 0.0024%
Character: 'X'   | Count:     112 | Percentage: 0.0100%
Character: 'Z'   | Count:     198 | Percentage: 0.0178%


# Stage 3 — Character Tokenizer

**Concept & Objectives:**

*Neural networks cannot process raw text strings like "ROMEO:". They operate strictly on numerical tensors (vectors and matrices). Tokenization is the process of translating raw text into integers (token IDs) and back again.*

# Core Concepts

**Vocabulary ($V$):**

1. The complete set of unique tokens the model knows. For our character-level model, $V$ contains $65$ unique characters (letters, numbers, punctuation, spaces, and newlines).

2. encode(str) -> list[int]: Maps each character in a string to its corresponding integer index in the vocabulary.

3. decode(list[int]) -> str: Maps each integer index back to its corresponding character string and joins them. *italicized text* *italicized text* *italicized text*

In [4]:
import torch

chars = sorted(list(set(text)))
vocab_size = len(chars)

#Create bidirectional lookup tables
#stoi : String to Integer (character -> integer)
#itos : Integer to String (integer -> character)


stoi = {ch: i for i, ch in enumerate(chars)}

itos = {i: ch for i, ch in enumerate(chars)}

#Encode function -> Converting String into a list of integers

def encode(s: str) -> list[int]:
  return [stoi[c] for c in s]


# Decode function: converts a list of integers back into a string
def decode(l: list[int]) -> str:
    return ''.join([itos[i] for i in l])


# Test string
sample_text = "ROMEO: Shall I hear more?"
encoded_sample = encode(sample_text)
decoded_sample = decode(encoded_sample)

print(f"Original Text : {sample_text!r}")
print(f"Encoded IDs   : {encoded_sample}")
print(f"Decoded Text  : {decoded_sample!r}")
print(f"Reconstruction Perfect? {sample_text == decoded_sample}")

# Convert entire dataset into a 1D PyTorch Tensor of 64-bit integers (torch.long)
data = torch.tensor(encode(text), dtype=torch.long)

print("\n--- DATA TENSOR SUMMARY ---")
print(f"Tensor Shape : {data.shape}")
print(f"Tensor Type  : {data.dtype}")
print(f"First 10 IDs : {data[:10].tolist()}")


Original Text : 'ROMEO: Shall I hear more?'
Encoded IDs   : [30, 27, 25, 17, 27, 10, 1, 31, 46, 39, 50, 50, 1, 21, 1, 46, 43, 39, 56, 1, 51, 53, 56, 43, 12]
Decoded Text  : 'ROMEO: Shall I hear more?'
Reconstruction Perfect? True

--- DATA TENSOR SUMMARY ---
Tensor Shape : torch.Size([1115394])
Tensor Type  : torch.int64
First 10 IDs : [18, 47, 56, 57, 58, 1, 15, 47, 58, 47]


In [7]:
# Check edge case text with newlines and punctuation
edge_case_text = "First Citizen:\n'To die, or to starve?'"

# Perform encoding and decoding
encoded = encode(edge_case_text)
decoded = decode(encoded)

assert decoded == edge_case_text, "Decoding mismatch detected!"

print("Tokenizer verification test passed successfully!")
print("Edge Case Input:\n", edge_case_text)
print("\nEncoded Integer Sequence:\n", encoded)

Tokenizer verification test passed successfully!
Edge Case Input:
 First Citizen:
'To die, or to starve?'

Encoded Integer Sequence:
 [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 5, 32, 53, 1, 42, 47, 43, 6, 1, 53, 56, 1, 58, 53, 1, 57, 58, 39, 56, 60, 43, 12, 5]


## Stage 4 — Train/Validation Split

**Concept & Objectives**

Before training our model, we must partition our dataset into two distinct parts:

*   **Training Data (~90%):** The portion of the text the model reads to update its internal weights via gradient descent. The model learns patterns, grammar, and vocabulary directly from this split.
*   **Validation Data (~10%):** A held-out portion of text that the model never trains on. We use it to compute validation loss during evaluation.

### Why We Need a Validation Set

*   **Overfitting Detection:** If a model has enough capacity, it can simply memorize the training text verbatim rather than learning general language patterns.
*   **Validation Loss:** By measuring performance on the validation set, we can check whether the model is actually generalizing to unseen Shakespeare text or just memorizing the training split. If training loss decreases while validation loss increases, the model is overfitting.
*   **Preventing Data Leakage:** Data leakage occurs when information from the validation or test set accidentally leaks into the training process. By strictly splitting the dataset chronologically or randomly before batching, we ensure the model never sees validation sequences during optimization.

In [8]:
import torch

# Define split ratio (90% training, 10% validation)
n = int(0.9 * len(data))

# Slice the 1D data tensor
train_data = data[:n]
val_data = data[n:]

print("--- DATA SPLIT SUMMARY ---")
print(f"Total tokens     : {len(data):,}")
print(f"Training tokens  : {len(train_data):,} ({len(train_data)/len(data)*100:.1f}%)")
print(f"Validation tokens: {len(val_data):,} ({len(val_data)/len(data)*100:.1f}%)")

--- DATA SPLIT SUMMARY ---
Total tokens     : 1,115,394
Training tokens  : 1,003,854 (90.0%)
Validation tokens: 111,540 (10.0%)


In [9]:
# Verify split continuity and non-overlap
print(f"Last token of train data ID: {train_data[-1].item()} ({itos[train_data[-1].item()]!r})")
print(f"First token of val data ID : {val_data[0].item()} ({itos[val_data[0].item()]!r})")
print(f"Total split check sum match: {len(train_data) + len(val_data) == len(data)}")

Last token of train data ID: 43 ('e')
First token of val data ID : 12 ('?')
Total split check sum match: True


## Stage 5 — Context Windows and Batches

**Concept & Objectives**

Now that we have our dataset split, we need to format the data into inputs ($X$) and targets ($Y$) that a Transformer can process.

### 1. Context Length ($T$ or `block_size`)

Language models do not process the entire 1-million-character dataset at once. Instead, they look at a fixed maximum window of previous characters called the **context window** or **block size** ($T$).

### 2. Why the Target is Shifted by One Character

Language modeling is **autoregressive next-token prediction**: given a sequence of characters, predict the character that comes next. For an input chunk $x$, target $y$ is the exact same chunk offset by 1 position to the right:

*   **Input ($X$):** "ROMEO"
*   **Target ($Y$):** "OMEO:"

Inside a context window of length $T$, there are actually $T$ individual training examples packed into one pass:
1. Given "R", predict 'O'
2. Given "RO", predict 'M'
3. Given "ROM", predict 'E'
4. Given "ROME", predict 'O'
5. Given "ROMEO", predict ':'

### 3. Batching ($B$ or `batch_size`)

GPUs achieve high computational throughput by processing multiple independent chunks in parallel. The **batch size** ($B$) specifies how many context chunks are fed into the network simultaneously.

### Key Dimension Notation

*   **$B$ (Batch Size):** Number of independent context sequences processed in parallel (e.g., $B = 4$).
*   **$T$ (Time / Context Length / Block Size):** Length of each context window (e.g., $T = 8$).
*   **$C$ (Channels / Vocabulary Size / Embedding Dim):** Vocabulary size (65) or hidden feature dimension.

In [11]:
import torch

# Explicitly set active device (cuda if GPU is available, else cpu)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Set seed for reproducibility
torch.manual_seed(1337)

# Hyperparameters for dataset batching
batch_size = 4  # B: How many independent sequences will we process in parallel?
block_size = 8  # T: What is the maximum context length for predictions?

def get_batch(split: str):
    # Select split
    data_source = train_data if split == 'train' else val_data

    # Generate B random starting indices from the data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))

    # Stack B context chunks of length T
    x = torch.stack([data_source[i:i+block_size] for i in ix])

    # Stack corresponding target chunks shifted by 1 index
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])

    # Move tensors to active GPU/CPU device
    x, y = x.to(device), y.to(device)
    return x, y

# Generate a sample batch
xb, yb = get_batch('train')

print("--- BATCH TENSOR SHAPES ---")
print(f"Input batch shape  (B, T): {xb.shape}")
print(f"Target batch shape (B, T): {yb.shape}")
print("\n--- SAMPLE INPUT TENSOR (xb) ---")
print(xb)
print("\n--- SAMPLE TARGET TENSOR (yb) ---")
print(yb)

--- BATCH TENSOR SHAPES ---
Input batch shape  (B, T): torch.Size([4, 8])
Target batch shape (B, T): torch.Size([4, 8])

--- SAMPLE INPUT TENSOR (xb) ---
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')

--- SAMPLE TARGET TENSOR (yb) ---
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')


## Stage 6 — Bigram Baseline

**Concept & Objectives**

Before building a complex Transformer, we construct the simplest possible language model: a **Bigram Language Model**. This baseline helps us evaluate whether our training pipeline works and provides a performance benchmark that our Transformer must beat.

### 1. What is a Bigram Model?

A bigram model predicts the next character based strictly on the identity of the current single character, completely ignoring all preceding context history.

*   If the current character is 'R', the model looks up a fixed probability distribution over what character typically follows 'R' in the training text.

### 2. Key Vocabulary & Concepts

*   **Logits ($z$):** Unnormalized raw output scores produced by the final neural network layer for each character in the vocabulary ($V = 65$). Higher scores mean higher likelihood. Shape: $(B, T, C)$.
*   **Probabilities ($P$):** Obtained by passing logits through the Softmax activation function:
    $$P(y_t = i) = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}$$
*   **Cross-Entropy Loss ($\mathcal{L}$):** Measures how closely our predicted probability distribution matches the actual target character.
*   **Theoretical Random Loss:** At initialization, if the model guesses completely at random with equal probability across $V = 65$ characters, the expected loss is:
    $$\mathcal{L}_{\text{initial}} = -\ln\left(\frac{1}{65}\right) \approx 4.1743$$

In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensors of integers
        logits = self.token_embedding_table(idx) # Shape: (B, T, C)

        if targets is None:
            loss = None
        else:
            # PyTorch F.cross_entropy expects shape (N, C) for logits and (N) for targets
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Get the predictions for the current indices
            logits, _ = self(idx)
            # Focus only on the last time step (the bigram prediction)
            logits = logits[:, -1, :] # Shape: (B, C)
            # Apply softmax to convert raw logits to probabilities
            probs = F.softmax(logits, dim=-1) # Shape: (B, C)
            # Sample from the probability distribution
            idx_next = torch.multinomial(probs, num_samples=1) # Shape: (B, 1)
            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # Shape: (B, T + 1)
        return idx

# Instantiate model
model = BigramLanguageModel(vocab_size).to(device)

# 1. Evaluate Untrained Loss
xb, yb = get_batch('train')
logits, initial_loss = model(xb, yb)
print(f"Untrained Initial Loss: {initial_loss.item():.4f} (Expected random baseline ~4.17)")

# 2. Generate text BEFORE training
context = torch.zeros((1, 1), dtype=torch.long, device=device) # Start with token 0 ('\n')
print("\n--- GENERATED TEXT BEFORE TRAINING ---")
print(decode(model.generate(context, max_new_tokens=100)[0].tolist()))

# 3. Train the Bigram Model
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
batch_size = 32

for iter in range(3000):
    # Sample a batch of data
    xb, yb = get_batch('train')

    # Evaluate loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"\nFinal Trained Loss: {loss.item():.4f}")

# 4. Generate text AFTER training
print("\n--- GENERATED TEXT AFTER TRAINING ---")
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))

Untrained Initial Loss: 4.7051 (Expected random baseline ~4.17)

--- GENERATED TEXT BEFORE TRAINING ---

pYCXxfRkRZd
wc'wfNfT;OLlTEeC K
jxqPToTb?bXAUG:C-SGJO-33SM:C?YI3a
hs:LVXJFhXeNuwqhObxZ.tSVrddXlaSZaNe

Final Trained Loss: 2.5265

--- GENERATED TEXT AFTER TRAINING ---

Wawice my.

Hastarom orou wabuts, tof is h ble mil ndill, ath iree sengmin lat Heriliovets, and Win nghir.
Swanousel lind me l.
HAshe ce hiry:
Sugr aisspllwhy.
Hentous n Boopetelaves
MPOLI s, d mothak


## Stage 7 — Self-Attention

**Concept & Objectives**

In the Bigram baseline, tokens were isolated and unable to look back at preceding context. **Self-Attention** is the core mechanism that allows tokens in a sequence to communicate with each other, dynamically gathering context from relevant prior tokens.

### 1. Intuition: Queries, Keys, and Values

Every token at every position projects its input representation into three distinct vectors:
*   **Query ($Q$):** "What am I looking for?" (Vector representing what the current token wants to know about its context).
*   **Key ($K$):** "What do I contain?" (Vector representing what identity/information this token offers to other tokens).
*   **Value ($V$):** "What information do I pass along?" (Vector containing the actual features to communicate if this token is attended to).

The interaction between a Query token at position $i$ and a Key token at position $j$ produces an attention score. If $Q_i$ and $K_j$ align well (high dot product), token $i$ pays high attention to token $j$ and incorporates more of $V_j$ into its representation.

### 2. Mathematical Formulation

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

*   **Dot Product ($Q K^T$):** Computes pairwise similarity between all queries and keys. Resulting shape: $(B, T, T)$.
*   **Scaling Factor ($\sqrt{d_k}$):** $d_k$ is the dimensionality of the head (`head_size`). Dividing by $\sqrt{d_k}$ prevents the dot products from growing excessively large, which would cause the softmax function to saturate and yield vanishing gradients.
*   **Causal Mask:** In autoregressive language modeling, token $t$ must never look into the future (tokens $> t$). We enforce this by setting future positions in the attention score matrix to $-\infty$ before taking the softmax, making their post-softmax probability exactly $0$.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

class SingleHeadAttention(nn.Module):
    """ One head of Causal Self-Attention """
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
        # Buffer for lower-triangular causal mask (not a trainable parameter)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x input shape: (B, T, C)
        B, T, C = x.shape
        
        # Project inputs to Keys, Queries, and Values
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)
        
        # Compute attention scores ("affinities")
        # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        
        # Apply causal mask: mask out future tokens by setting them to -inf
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        
        # Normalize scores to probabilities
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        
        # Perform weighted aggregation of values
        out = wei @ v # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out, wei

# Test setup
B, T, C = 2, 8, 32 # Batch size=2, Context length=8, Embedding dim=32
head_size = 16    # Attention head size = 16

# Dummy input tensor representing embedded tokens
x = torch.randn(B, T, C, device=device)

# Instantiate single head
head = SingleHeadAttention(n_embd=C, head_size=head_size, block_size=T).to(device)
out, attention_weights = head(x)

print("--- TENSOR DIMENSIONS ---")
print(f"Input shape  (B, T, C)         : {x.shape}")
print(f"Attention Matrix (B, T, T)     : {attention_weights.shape}")
print(f"Output shape (B, T, head_size) : {out.shape}")

## Stage 8 — Multi-Head Self-Attention

**Concept & Objectives**

In Stage 7, we built a single head of causal self-attention. A single head allows tokens to communicate, but it can only focus on one type of relationship at a time (e.g., finding which noun a pronoun refers to).

### 1. Why Multiple Heads?

Language is complex and requires tracking multiple linguistic relationships simultaneously:
*   **Head 1** might learn to track who is speaking (ROMEO: $\rightarrow$ dialogue line).
*   **Head 2** might learn to track rhyme patterns or poetic meter.
*   **Head 3** might learn to track grammatical subject-verb agreement.
*   **Head 4** might learn to track punctuation boundaries.

### 2. How Multi-Head Attention Works

Instead of using one large attention head of size $C$, we run $h$ smaller attention heads in parallel:

*   **Sub-spaces:** Each head has a feature dimension of $d_{\text{head}} = \frac{C}{h}$. Each head operates independently in its own feature subspace.
*   **Concatenation:** The output vectors of all $h$ heads are concatenated along the channel dimension back to size $C$.
*   **Final Projection:** A final linear projection layer combines the outputs from all heads so information can mix across channels.

In [ ]:
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

class MultiHeadAttention(nn.Module):
    """ Multiple heads of Causal Self-Attention in parallel """
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout=0.0):
        super().__init__()
        # Instantiate h independent attention heads
        self.heads = nn.ModuleList([
            SingleHeadAttention(n_embd, head_size, block_size, dropout) 
            for _ in range(num_heads)
        ])
        # Projection layer to mix representations from all heads
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run input x through each head in parallel and unpack only the output tensor [0]
        # Each head output shape: (B, T, head_size)
        head_outputs = [h(x)[0] for h in self.heads]
        
        # Concatenate outputs along the channel dimension
        # (B, T, num_heads * head_size) -> (B, T, n_embd)
        out = torch.cat(head_outputs, dim=-1)
        
        # Apply linear projection and dropout
        out = self.dropout(self.proj(out))
        return out

# Hyperparameters
B, T, C = 2, 8, 32    # Batch=2, Context Length=8, Embedding Dim=32
num_heads = 4         # h = 4 parallel heads
head_size = C // num_heads  # head_size = 32 / 4 = 8

# Create dummy input tensor
x = torch.randn(B, T, C, device=device)

# Instantiate Multi-Head Attention block
mha = MultiHeadAttention(num_heads=num_heads, head_size=head_size, n_embd=C, block_size=T).to(device)
out = mha(x)

print("--- MULTI-HEAD ATTENTION TENSOR SHAPES ---")
print(f"Input shape (B, T, C)             : {x.shape}")
print(f"Number of Heads (h)               : {num_heads}")
print(f"Dimension per Head (d_head)       : {head_size}")
print(f"Concatenated & Projected Output   : {out.shape}")

## Stage 9 — Feed-Forward Network

**Concept & Objectives**

If Multi-Head Attention is how tokens **communicate** with each other, the Feed-Forward Network (FFN) is where tokens **process** and "think" about the information they just gathered.

### 1. Intuition: Communication vs. Computation

*   **Self-Attention:** A communication step. Tokens look around at each other and aggregate context vectors. It is a linear combination of token vectors across time.
*   **Feed-Forward Network:** A computation step. Applied to every token position independently and identically. Tokens do not communicate across time in this step; instead, each token vector is passed through a small Multi-Layer Perceptron (MLP) to transform its internal features.

### 2. Structure of the Feed-Forward Block

A standard Transformer feed-forward network consists of two linear transformations with a non-linear activation function in between:

*   **Linear Expansion Layer:** Projects the embedding dimension $C$ (`n_embd`) to a higher-dimensional space (typically $4 \times C$).
*   **Non-Linear Activation:** Enables the network to learn complex, non-linear relationships (e.g., `nn.ReLU()` or `nn.GELU()`).
*   **Linear Contraction Layer:** Projects the high-dimensional representation back down to dimension $C$.
*   **Dropout:** Regularization layer to prevent overfitting.

In [ ]:
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

class FeedForward(nn.Module):
    """ A simple linear layer followed by a non-linearity """
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            # Expand dimension by 4x
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            # Project back down to original dimension
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x shape: (B, T, C)
        return self.net(x)

# Hyperparameters
B, T, C = 2, 8, 32  # Batch=2, Context Length=8, Embedding Dim=32

# Create dummy input tensor
x = torch.randn(B, T, C, device=device)

# Instantiate FeedForward module
ffn = FeedForward(n_embd=C).to(device)
out = ffn(x)

print("--- FEED-FORWARD TENSOR SHAPES ---")
print(f"Input shape  (B, T, C) : {x.shape}")
print(f"Output shape (B, T, C) : {out.shape}")